# Pipeline Final de Produção - Anti-Money Laundering Detection

**PIPELINE METODOLOGICAMENTE CORRETO**

Este notebook implementa um pipeline completo e reprodutível usando:
- ✅ **sklearn.pipeline.Pipeline** para transformações
- ✅ **imblearn.pipeline.Pipeline** para RUS (Random Under Sampling)
- ✅ **Caminhos dinâmicos** via source/config.py (zero hardcoding)
- ✅ **Zero Data Leakage** (fit apenas em treino)
- ✅ **Reprodutibilidade total** (seeds, paths relativos)

**Autor:** TCC - Anti-Money Laundering Detection  
**Data:** Janeiro 2026  
**Metodologia:** Scikit-Learn Production Pipelines + Imbalanced-Learn Strategy

## 📋 Índice

1. [Setup e Imports](#1-setup-e-imports)
2. [Carregamento de Dados](#2-carregamento-de-dados)
3. [Análise Exploratória Rápida](#3-análise-exploratória-rápida)
4. [Divisão Treino/OOT](#4-divisão-treino-oot)
5. [Pipeline de Pré-processamento](#5-pipeline-de-pré-processamento)
6. [Pipeline Completo com RUS](#6-pipeline-completo-com-rus)
7. [Treinamento de Múltiplos Modelos](#7-treinamento-de-múltiplos-modelos)
8. [Avaliação e Comparação](#8-avaliação-e-comparação)
9. [Persistência do Pipeline](#9-persistência-do-pipeline)
10. [Inferência em Produção](#10-inferência-em-produção)

## 1. Setup e Imports

**CRÍTICO:** Todos os caminhos vêm de `source/config.py` - ZERO HARDCODING

In [ ]:
# Adicionar source ao path do Python
import sys
from pathlib import Path

# Detectar se estamos em notebook (notebooks/) ou root
if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))
else:
    sys.path.insert(0, str(Path.cwd()))

# Imports padrão
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Scikit-Learn - Pipelines e Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# Imbalanced-Learn - Pipeline com RUS
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb

# Métricas
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, accuracy_score, precision_score,
    recall_score, average_precision_score, make_scorer
)

# Category Encoders
from category_encoders import TargetEncoder

# Persistência
import joblib

# Config do projeto - CAMINHOS DINÂMICOS
from source.config import (
    PROJ_ROOT, DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, EXTERNAL_DATA_DIR,
    MODELS_DIR, FIGURES_DIR, get_data_path, get_model_path, get_figure_path
)

# Logging
from loguru import logger

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Seed global para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

logger.success("✅ Imports completos!")
logger.info(f"📁 Projeto: {PROJ_ROOT}")
logger.info(f"📁 Dados processados: {PROCESSED_DATA_DIR}")
logger.info(f"📁 Modelos: {MODELS_DIR}")

## 2. Carregamento de Dados

**CRÍTICO:** Usando `get_data_path()` do config.py (não hardcoded paths)

In [ ]:
# Carregar dados processados (gerados por source/dataset.py)
logger.info("Carregando dados...")

# Verificar se existem dados já preparados
treino_path = get_data_path('df_treino.csv', 'processed')
oot_path = get_data_path('df_oot.csv', 'processed')

if not treino_path.exists() or not oot_path.exists():
    logger.error("❌ Dados processados não encontrados!")
    logger.info("Execute primeiro: python source/dataset.py")
    raise FileNotFoundError(f"Arquivos não encontrados: {treino_path}, {oot_path}")

# Carregar
df_treino = pd.read_csv(treino_path)
df_oot = pd.read_csv(oot_path)

logger.success(f"✅ Dados carregados!")
logger.info(f"  Treino: {df_treino.shape}")
logger.info(f"  OOT:    {df_oot.shape}")

# Exibir primeiras linhas
display(df_treino.head())

## 3. Análise Exploratória Rápida

Verificar qualidade dos dados e distribuição do target.

In [ ]:
# Identificar target
TARGET_COL = 'Is Laundering'

if TARGET_COL not in df_treino.columns:
    logger.error(f"❌ Coluna target '{TARGET_COL}' não encontrada!")
    logger.info(f"Colunas disponíveis: {df_treino.columns.tolist()}")
    raise ValueError(f"Target {TARGET_COL} não existe")

# Estatísticas básicas
logger.info("="*80)
logger.info("ANÁLISE EXPLORATÓRIA")
logger.info("="*80)

# Missing values
missing_treino = df_treino.isnull().sum()
missing_treino = missing_treino[missing_treino > 0].sort_values(ascending=False)

if len(missing_treino) > 0:
    logger.warning(f"⚠️  {len(missing_treino)} colunas com valores ausentes:")
    for col, count in missing_treino.items():
        pct = (count / len(df_treino)) * 100
        logger.info(f"  {col}: {count} ({pct:.2f}%)")
else:
    logger.success("✅ Nenhum valor ausente!")

# Distribuição do target
logger.info("\n" + "="*80)
logger.info("DISTRIBUIÇÃO DO TARGET")
logger.info("="*80)

for dataset_name, df in [('Treino', df_treino), ('OOT', df_oot)]:
    counts = df[TARGET_COL].value_counts()
    pct_fraud = (counts.get(1, 0) / len(df)) * 100
    logger.info(f"\n{dataset_name}:")
    logger.info(f"  Total: {len(df):,}")
    logger.info(f"  Legítimas (0): {counts.get(0, 0):,} ({100-pct_fraud:.2f}%)")
    logger.info(f"  Fraude (1):    {counts.get(1, 0):,} ({pct_fraud:.2f}%)")

# Visualizar distribuição
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (dataset_name, df) in enumerate([('Treino', df_treino), ('OOT', df_oot)]):
    counts = df[TARGET_COL].value_counts()
    axes[idx].bar(['Legítima', 'Fraude'], counts.values, color=['steelblue', 'coral'], edgecolor='black')
    axes[idx].set_title(f'Distribuição - {dataset_name}', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Frequência', fontsize=11)
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Anotar percentuais
    for i, v in enumerate(counts.values):
        pct = (v / len(df)) * 100
        axes[idx].text(i, v, f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(get_figure_path('distribuicao_target.png'), dpi=300, bbox_inches='tight')
plt.show()

logger.success("✅ Análise exploratória concluída!")

## 4. Divisão Treino/OOT

Separar features (X) e target (y) em treino e OOT.

In [ ]:
# Separar X e y
logger.info("Separando features (X) e target (y)...")

X_train = df_treino.drop(columns=[TARGET_COL])
y_train = df_treino[TARGET_COL]

X_oot = df_oot.drop(columns=[TARGET_COL])
y_oot = df_oot[TARGET_COL]

logger.success("✅ Separação concluída!")
logger.info(f"  X_train: {X_train.shape}")
logger.info(f"  y_train: {y_train.shape} - {y_train.value_counts().to_dict()}")
logger.info(f"  X_oot:   {X_oot.shape}")
logger.info(f"  y_oot:   {y_oot.shape} - {y_oot.value_counts().to_dict()}")

# Identificar tipos de colunas
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

logger.info(f"\n📊 Tipos de features:")
logger.info(f"  Numéricas:    {len(numeric_cols)}")
logger.info(f"  Categóricas:  {len(categorical_cols)}")

# Mostrar algumas
if numeric_cols:
    logger.info(f"\nExemplos numéricos: {numeric_cols[:5]}")
if categorical_cols:
    logger.info(f"Exemplos categóricos: {categorical_cols[:5]}")

## 5. Pipeline de Pré-processamento

**CRÍTICO:** Pipeline usando `sklearn.compose.ColumnTransformer` para transformações separadas por tipo.

**Estratégia:**
- **Numéricas:** SimpleImputer (mediana) → Log+1 → StandardScaler
- **Categóricas (baixa cardinalidade):** SimpleImputer (most_frequent) → OneHotEncoder
- **Categóricas (alta cardinalidade):** SimpleImputer (most_frequent) → TargetEncoder

In [ ]:
# Classificar categóricas por cardinalidade
logger.info("Classificando variáveis categóricas por cardinalidade...")

onehot_cols = []
target_encoding_cols = []

for col in categorical_cols:
    n_unique = X_train[col].nunique()
    
    if n_unique <= 10:
        onehot_cols.append(col)
    else:
        target_encoding_cols.append(col)

logger.info(f"  One-Hot Encoding (≤10 categorias):  {len(onehot_cols)}")
if onehot_cols:
    logger.info(f"    {onehot_cols}")

logger.info(f"  Target Encoding (>10 categorias):   {len(target_encoding_cols)}")
if target_encoding_cols:
    logger.info(f"    {target_encoding_cols[:5]}...")

In [ ]:
# Transformador customizado para log(x+1)
class LogTransformer(BaseEstimator, TransformerMixin):
    """Aplica transformação log(x+1) em features numéricas."""
    
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return np.log1p(X)
    
    def get_feature_names_out(self, input_features=None):
        return input_features


# Pipeline para features numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log', LogTransformer()),
    ('scaler', StandardScaler())
])

# Pipeline para categóricas (One-Hot)
onehot_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Pipeline para categóricas (Target Encoding)
target_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target', TargetEncoder(smoothing=1.0, min_samples_leaf=10))
])

# ColumnTransformer combinando tudo
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat_onehot', onehot_transformer, onehot_cols),
        ('cat_target', target_transformer, target_encoding_cols)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

logger.success("✅ Pipeline de pré-processamento criado!")
logger.info(f"  Transformadores: 3 (numeric, onehot, target)")
logger.info(f"  Features totais: {len(numeric_cols) + len(onehot_cols) + len(target_encoding_cols)}")

## 6. Pipeline Completo com RUS (Random Under Sampling)

**METODOLOGIA CRÍTICA:**
- Usamos `imblearn.pipeline.Pipeline` (não `sklearn.pipeline.Pipeline`)
- RUS é aplicado APENAS durante `.fit()` no treino
- Durante `.predict()` ou `.transform()`, RUS é bypassed
- Isso previne Data Leakage no conjunto OOT

In [ ]:
# Função para criar pipeline completo com modelo
def create_full_pipeline(model, use_rus=True, random_state=RANDOM_STATE):
    """
    Cria pipeline completo: Preprocessor → RUS → Model
    
    Args:
        model: Modelo sklearn-compatible
        use_rus: Se True, aplica Random Under Sampling
        random_state: Seed para reprodutibilidade
    
    Returns:
        imblearn.pipeline.Pipeline
    """
    steps = [('preprocessor', preprocessor)]
    
    if use_rus:
        rus = RandomUnderSampler(
            sampling_strategy='auto',
            random_state=random_state
        )
        steps.append(('rus', rus))
    
    steps.append(('model', model))
    
    return ImbPipeline(steps=steps)


# Teste: criar pipeline com Logistic Regression
logger.info("Testando criação de pipeline...")

test_pipeline = create_full_pipeline(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    use_rus=True
)

logger.success("✅ Pipeline completo criado!")
logger.info(f"  Steps: {[step[0] for step in test_pipeline.steps]}")
logger.info(f"  RUS incluído: Sim")
logger.info(f"  Modelo: LogisticRegression")

## 7. Treinamento de Múltiplos Modelos

Treinar 5 algoritmos com o mesmo pipeline (garantindo comparação justa).

In [ ]:
# Definir modelos a treinar
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, 
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=RANDOM_STATE
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric='logloss'
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )
}

logger.info(f"🎯 Treinando {len(models)} modelos com RUS...")

In [ ]:
# Treinar todos os modelos
trained_pipelines = {}
training_times = {}

for model_name, model in models.items():
    logger.info(f"\n{'='*80}")
    logger.info(f"Treinando: {model_name}")
    logger.info(f"{'='*80}")
    
    # Criar pipeline
    pipeline = create_full_pipeline(model, use_rus=True)
    
    # Treinar
    start_time = datetime.now()
    
    # CRITICAL: Fit com X_train e y_train
    # RUS será aplicado automaticamente APENAS aqui
    pipeline.fit(X_train, y_train)
    
    end_time = datetime.now()
    training_time = (end_time - start_time).total_seconds()
    
    # Armazenar
    trained_pipelines[model_name] = pipeline
    training_times[model_name] = training_time
    
    logger.success(f"✅ {model_name} treinado em {training_time:.2f}s")

logger.success(f"\n🎉 Todos os {len(trained_pipelines)} modelos treinados!")

## 8. Avaliação e Comparação

Avaliar todos os modelos em treino e OOT.

In [ ]:
# Função de avaliação
def evaluate_model(pipeline, X, y, dataset_name='test'):
    """
    Avalia modelo e retorna métricas.
    
    Args:
        pipeline: Pipeline treinado
        X: Features
        y: Target real
        dataset_name: Nome do dataset
    
    Returns:
        Dict com métricas
    """
    # Predições
    y_pred = pipeline.predict(X)
    y_proba = pipeline.predict_proba(X)[:, 1]
    
    # Métricas
    metrics = {
        'dataset': dataset_name,
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred, zero_division=0),
        'f1': f1_score(y, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y, y_proba),
        'avg_precision': average_precision_score(y, y_proba)
    }
    
    return metrics


# Avaliar todos os modelos
results = []

for model_name, pipeline in trained_pipelines.items():
    logger.info(f"\nAvaliando {model_name}...")
    
    # Treino
    train_metrics = evaluate_model(pipeline, X_train, y_train, 'train')
    train_metrics['model'] = model_name
    train_metrics['training_time'] = training_times[model_name]
    results.append(train_metrics)
    
    # OOT
    oot_metrics = evaluate_model(pipeline, X_oot, y_oot, 'oot')
    oot_metrics['model'] = model_name
    oot_metrics['training_time'] = training_times[model_name]
    results.append(oot_metrics)
    
    logger.info(f"  Treino - F1: {train_metrics['f1']:.4f} | ROC-AUC: {train_metrics['roc_auc']:.4f}")
    logger.info(f"  OOT    - F1: {oot_metrics['f1']:.4f} | ROC-AUC: {oot_metrics['roc_auc']:.4f}")

# Converter para DataFrame
df_results = pd.DataFrame(results)

logger.success("✅ Avaliação concluída!")

In [ ]:
# Exibir resultados
logger.info("\n" + "="*80)
logger.info("RESULTADOS CONSOLIDADOS")
logger.info("="*80)

# Separar por dataset
df_train = df_results[df_results['dataset'] == 'train'].copy()
df_oot = df_results[df_results['dataset'] == 'oot'].copy()

# Ordenar por ROC-AUC (OOT)
df_oot_sorted = df_oot.sort_values('roc_auc', ascending=False)

logger.info("\n📊 RANKING - Validação OOT (ordenado por ROC-AUC):")
display(df_oot_sorted[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'training_time']])

# Melhor modelo
best_model_name = df_oot_sorted.iloc[0]['model']
best_roc_auc = df_oot_sorted.iloc[0]['roc_auc']

logger.success(f"\n🏆 MELHOR MODELO: {best_model_name} (ROC-AUC OOT: {best_roc_auc:.4f})")

In [ ]:
# Visualizar comparação
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    # Dados
    train_values = df_train.sort_values('model')[metric].values
    oot_values = df_oot.sort_values('model')[metric].values
    model_names = df_train.sort_values('model')['model'].values
    
    # Plot
    x = np.arange(len(model_names))
    width = 0.35
    
    ax.bar(x - width/2, train_values, width, label='Treino', color='steelblue', edgecolor='black')
    ax.bar(x + width/2, oot_values, width, label='OOT', color='coral', edgecolor='black')
    
    ax.set_ylabel(metric.upper(), fontsize=12)
    ax.set_title(f'{metric.upper()} - Comparação Treino vs OOT', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(get_figure_path('comparacao_modelos.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Salvar resultados
results_train_path = get_data_path('model_results_train_final.csv', 'processed')
results_oot_path = get_data_path('model_results_oot_final.csv', 'processed')

df_train.to_csv(results_train_path, index=False)
df_oot.to_csv(results_oot_path, index=False)

logger.success(f"✅ Resultados salvos:")
logger.info(f"  {results_train_path}")
logger.info(f"  {results_oot_path}")

## 9. Persistência do Pipeline

Salvar o melhor pipeline para produção.

In [ ]:
# Salvar melhor pipeline
best_pipeline = trained_pipelines[best_model_name]

pipeline_path = get_model_path(f'pipeline_final_{best_model_name.lower().replace(" ", "_")}.pkl')
joblib.dump(best_pipeline, pipeline_path)

logger.success(f"✅ Pipeline salvo: {pipeline_path}")

# Salvar metadados
metadata = {
    'timestamp': datetime.now().isoformat(),
    'best_model': best_model_name,
    'roc_auc_oot': float(best_roc_auc),
    'random_state': RANDOM_STATE,
    'use_rus': True,
    'numeric_features': numeric_cols,
    'onehot_features': onehot_cols,
    'target_encoding_features': target_encoding_cols,
    'all_results': df_results.to_dict(orient='records')
}

import json
metadata_path = get_model_path('pipeline_final_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

logger.success(f"✅ Metadados salvos: {metadata_path}")

## 10. Inferência em Produção

Demonstrar como carregar e usar o pipeline salvo.

In [ ]:
# Simular ambiente de produção
logger.info("="*80)
logger.info("SIMULAÇÃO DE PRODUÇÃO")
logger.info("="*80)

# Carregar pipeline salvo
loaded_pipeline = joblib.load(pipeline_path)
logger.success(f"✅ Pipeline carregado de: {pipeline_path}")

# Pegar amostra do OOT para teste
sample_size = 10
sample_indices = np.random.choice(X_oot.index, size=sample_size, replace=False)
X_sample = X_oot.loc[sample_indices]
y_sample = y_oot.loc[sample_indices]

# Predizer
predictions = loaded_pipeline.predict(X_sample)
probabilities = loaded_pipeline.predict_proba(X_sample)[:, 1]

# Mostrar resultados
inference_results = pd.DataFrame({
    'Real': y_sample.values,
    'Predito': predictions,
    'Prob_Fraude': probabilities,
    'Correto': (y_sample.values == predictions)
})

logger.info("\n📊 Resultados da Inferência (amostra de 10):")
display(inference_results)

accuracy_sample = (inference_results['Correto'].sum() / len(inference_results)) * 100
logger.info(f"\nAcurácia na amostra: {accuracy_sample:.1f}%")

## ✅ Conclusão

### Pipeline Implementado com Sucesso

**Garantias Metodológicas:**
1. ✅ **Zero Hardcoding** - Todos os caminhos via `source/config.py`
2. ✅ **Zero Data Leakage** - Fit apenas em treino, transform em OOT
3. ✅ **RUS Correto** - Via `imblearn.pipeline` (ativo apenas no fit)
4. ✅ **Reprodutibilidade** - Seeds fixas, paths relativos
5. ✅ **Modularidade** - `sklearn.pipeline.Pipeline` para transformações
6. ✅ **Persistência** - Pipeline completo salvo em `.pkl`

**Próximos Passos:**
1. Ajustar hiperparâmetros do melhor modelo (GridSearchCV)
2. Implementar validação cruzada temporal
3. Analisar feature importance
4. Deploy via API (Flask/FastAPI)
5. Monitoramento de drift em produção

**Arquivos Gerados:**
- `{pipeline_path.name}` - Pipeline completo
- `pipeline_final_metadata.json` - Metadados
- `model_results_*_final.csv` - Resultados detalhados
- `*.png` - Visualizações

---

**Metodologia:** Scikit-Learn Production Pipelines + Imbalanced-Learn Strategy  
**Data:** Janeiro 2026  
**Status:** ✅ Pronto para Produção